In [1]:
import sys
sys.path.append("..")

from model.simulate import run_plan
from model.optimize import optimize_country

print(optimize_country("USA"))

[204, 204, 204, 204, 204, 204, 204, 204, 204, 289, 289, 289]


In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv("../.env")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

llm.invoke("Say hello in five words.").content

'Hello! How are you today?'

In [3]:
import sys
sys.path.append("..")

import json
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from model.simulate import run_plan, salaries, staff, forecast, FIRE_COST
from model.optimize import optimize_country

load_dotenv("../.env")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Setup done")




plan_df = pd.read_csv("../data/processed/plan.csv")

def country_facts(country):
    plan = plan_df[plan_df["country"] == country].sort_values("month")["agents"].tolist()
    best = run_plan(country, plan)
    keep = run_plan(country, [int(staff[country])] * 12)
    fc = forecast[forecast["country"] == country]
    salary = float(salaries[country])
    budget = float(fc["avg_eligible_budget"].iloc[0])
    net_best = best["revenue"].sum() - best["cost"].sum()
    net_keep = keep["revenue"].sum() - keep["cost"].sum()
    value_per_agent = 10 * 365 / 60 * 0.135 * budget

    return {
        "country": country,
        "agents_today": int(staff[country]),
        "recommended_jan_sep": int(plan[0]),
        "recommended_oct_dec": int(plan[9]),
        "agents_fired_in_january": int(best["fires"].sum()),
        "agents_hired_during_year": int(best["hires"].sum()),
        "one_time_firing_cost_usd": round(best["fires"].sum() * salary * (FIRE_COST + 1/12)),
        "net_value_recommended_usd_m": round(net_best / 1e6, 2),
        "net_value_if_unchanged_usd_m": round(net_keep / 1e6, 2),
        "net_value_gain_usd_m": round((net_best - net_keep) / 1e6, 2),
        "avg_daily_eligible_signups": round(float(fc["daily_eligible_base"].mean()), 1),
        "busiest_month": int(fc.loc[fc["daily_eligible_base"].idxmax(), "month"]),
        "max_avg_wait_days": round(float(best["avg_wait_days"].max()), 1),
        "advertisers_abandoned": round(float(best["abandoned"].sum())),
        "avg_utilization_pct": round(float(best["utilization"].mean() * 100)),
        "annual_salary_usd": round(salary),
        "avg_eligible_budget_usd": round(budget),
        "value_per_agent_vs_salary": round(value_per_agent / salary, 2),
    }

print(json.dumps(country_facts("Vietnam"), indent=2))

Setup done
{
  "country": "Vietnam",
  "agents_today": 59,
  "recommended_jan_sep": 77,
  "recommended_oct_dec": 107,
  "agents_fired_in_january": 0,
  "agents_hired_during_year": 48,
  "one_time_firing_cost_usd": 0,
  "net_value_recommended_usd_m": 9.78,
  "net_value_if_unchanged_usd_m": 3.2,
  "net_value_gain_usd_m": 6.57,
  "avg_daily_eligible_signups": 13.3,
  "busiest_month": 11,
  "max_avg_wait_days": 0.3,
  "advertisers_abandoned": 0,
  "avg_utilization_pct": 85,
  "annual_salary_usd": 17904,
  "avg_eligible_budget_usd": 17303,
  "value_per_agent_vs_salary": 7.94
}


In [4]:
from pydantic import BaseModel, Field

class MemoText(BaseModel):
    headline: str = Field(description="One sentence: the recommendation for this country.")
    reasons: list[str] = Field(description="Exactly 3 short reasons, each citing numbers from the facts.")
    risks: list[str] = Field(description="2 to 3 risks or caveats of the recommendation.")
    decision_needed: str = Field(description="One sentence: what the approver must decide or sign off on.")

In [5]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You write short staffing approval memos for Google Ads advertiser support.\n"
     "Rules:\n"
     "- Use ONLY numbers that appear in the facts. Never calculate, estimate or invent numbers.\n"
     "- Write money as $X.XM for millions (e.g. net_value_gain_usd_m 6.57 -> $6.57M) and $X,XXX otherwise.\n"
     "- Never show field names like _usd_m in the text.\n"
     "- The headline and decision must mention BOTH headcounts: Jan-Sep and Oct-Dec.\n"
     "- Reasons must explain WHY, not just restate numbers. Use these where relevant: "
     "value_per_agent_vs_salary (above 1 means each agent earns more than they cost); "
     "firing early is cheaper than paying idle agents all year; "
     "Q4 agents must be hired in September because of the 1-month ramp-up; "
     "how many advertisers would wait or be lost.\n"
     "- Plain language for a busy manager, no jargon."),
    ("human",
     "Facts for {country} (from our 2026 staffing model):\n{facts}\n\n"
     "Background rules: each agent supports up to 10 advertisers for 60 days; "
     "new hires take 1 month to ramp up; firing costs 40% of annual salary plus 1 month notice; "
     "peak demand is in Q4."),
])

In [6]:
chain = prompt | llm.with_structured_output(MemoText)

facts = country_facts("Vietnam")
text = chain.invoke({"country": "Vietnam", "facts": json.dumps(facts, indent=2)})
text

MemoText(headline='Approve staffing increase from 77 agents Jan-Sep to 107 agents Oct-Dec in Vietnam.', reasons=['Hiring 107 agents in Q4 is crucial to meet peak demand, as the busiest month is November.', 'Each agent generates significant value, with a value per agent of 7.94, far exceeding their salary of $17,904.', 'Delaying hiring until October would risk losing advertisers, as the average wait time is only 0.3 days.'], risks=['If hiring is delayed, we may not have enough agents to handle the increased workload in Q4.', 'There is a risk of overstaffing if advertiser demand does not meet expectations, but this is mitigated by the high value per agent.', 'While there are no firing costs currently, future adjustments may incur costs if we need to reduce headcount.'], decision_needed='Please approve the staffing increase to 107 agents for Q4.')

In [7]:
def format_memo(facts, text):
    lines = [
        f"## {facts['country']}: {text.headline}", "",
        "| | Today | Recommended |",
        "|---|---|---|",
        f"| Agents Jan–Sep | {facts['agents_today']} | {facts['recommended_jan_sep']} |",
        f"| Agents Oct–Dec | {facts['agents_today']} | {facts['recommended_oct_dec']} |",
        f"| Net value 2026 | ${facts['net_value_if_unchanged_usd_m']}M | ${facts['net_value_recommended_usd_m']}M |",
        "", "**Why:**",
    ]
    lines += [f"- {r}" for r in text.reasons]
    lines += ["", "**Risks:**"]
    lines += [f"- {r}" for r in text.risks]
    lines += ["", f"**Decision needed:** {text.decision_needed}", ""]
    return "\n".join(lines)

print(format_memo(facts, text))

## Vietnam: Approve staffing increase from 77 agents Jan-Sep to 107 agents Oct-Dec in Vietnam.

| | Today | Recommended |
|---|---|---|
| Agents Jan–Sep | 59 | 77 |
| Agents Oct–Dec | 59 | 107 |
| Net value 2026 | $3.2M | $9.78M |

**Why:**
- Hiring 107 agents in Q4 is crucial to meet peak demand, as the busiest month is November.
- Each agent generates significant value, with a value per agent of 7.94, far exceeding their salary of $17,904.
- Delaying hiring until October would risk losing advertisers, as the average wait time is only 0.3 days.

**Risks:**
- If hiring is delayed, we may not have enough agents to handle the increased workload in Q4.
- There is a risk of overstaffing if advertiser demand does not meet expectations, but this is mitigated by the high value per agent.
- While there are no firing costs currently, future adjustments may incur costs if we need to reduce headcount.

**Decision needed:** Please approve the staffing increase to 107 agents for Q4.



In [8]:
import os
os.makedirs("../report", exist_ok=True)

memos = ["# 2026 Staffing Approval Memos", ""]
for country in sorted(plan_df["country"].unique()):
    facts = country_facts(country)
    text = chain.invoke({"country": country, "facts": json.dumps(facts, indent=2)})
    memos.append(format_memo(facts, text))
    print("done:", country)

with open("../report/staffing_memos.md", "w", encoding="utf-8") as f:
    f.write("\n".join(memos))
print("Saved report/staffing_memos.md")

done: Argentina
done: Australia
done: Brazil
done: Canada
done: China
done: France
done: Germany
done: India
done: Indonesia
done: Italy
done: Japan
done: Mexico
done: Poland
done: South Africa
done: South Korea
done: Spain
done: Thailand
done: Turkey
done: UAE
done: UK
done: USA
done: Vietnam


PermissionError: [Errno 13] Permission denied: '../report/staffing_memos.md'

In [9]:
from langchain_core.tools import tool

summary_df = pd.read_csv("../data/processed/plan_summary.csv")
VALID = sorted(plan_df["country"].unique())

def summarize(country, plan, scenario):
    """Run the simulator and return the key results as plain numbers."""
    m = run_plan(country, plan, scenario)
    return {
        "country": country, "scenario": scenario,
        "agents_jan_sep": int(plan[0]), "agents_oct_dec": int(plan[9]),
        "net_value_usd_m": float(round((m["revenue"].sum() - m["cost"].sum()) / 1e6, 2)),
        "revenue_usd_m": float(round(m["revenue"].sum() / 1e6, 2)),
        "cost_usd_m": float(round(m["cost"].sum() / 1e6, 2)),
        "advertisers_served": int(round(m["served"].sum())),
        "advertisers_abandoned": int(round(m["abandoned"].sum())),
        "worst_month_avg_wait_days": float(round(m["avg_wait_days"].max(), 1)),
        "still_waiting_end_of_year": int(round(m["waiting_end"].iloc[-1])),
    }

def check_country(country):
    if country not in VALID:
        return {"error": f"Unknown country '{country}'. Valid names: {VALID}"}
    return None

@tool
def get_recommended_plan(country: str, scenario: str = "base") -> dict:
    """Get our model's recommended 2026 headcount for a country and its simulated results.
    scenario: 'base', 'low' (15% fewer sign-ups) or 'high' (15% more sign-ups)."""
    if err := check_country(country):
        return err
    plan = plan_df[plan_df["country"] == country].sort_values("month")["agents"].tolist()
    return summarize(country, plan, scenario)

@tool
def evaluate_plan(country: str, agents_jan_sep: int, agents_oct_dec: int, scenario: str = "base") -> dict:
    """Simulate a headcount someone proposes: agents_jan_sep agents for January-September and
    agents_oct_dec for October-December. For a flat plan, use the same number twice.
    scenario: 'base', 'low' or 'high'."""
    if err := check_country(country):
        return err
    plan = [agents_jan_sep] * 9 + [agents_oct_dec] * 3
    return summarize(country, plan, scenario)

@tool
def reoptimize(country: str, scenario: str) -> dict:
    """Find the best headcount for a country under a different demand scenario ('low' or 'high').
    Use this when asked what the plan SHOULD be if sign-ups change. Takes about 10 seconds."""
    if err := check_country(country):
        return err
    plan = optimize_country(country, scenario)
    return summarize(country, plan, scenario)

@tool
def all_countries_overview() -> list:
    """Recommended plan and net value gain for all 22 countries. Use for rankings, totals or comparisons."""
    return summary_df.to_dict(orient="records")

tools = [get_recommended_plan, evaluate_plan, reoptimize, all_countries_overview]

# quick test, with no AI involved yet
evaluate_plan.invoke({"country": "France", "agents_jan_sep": 150, "agents_oct_dec": 150})

{'country': 'France',
 'scenario': 'base',
 'agents_jan_sep': 150,
 'agents_oct_dec': 150,
 'net_value_usd_m': 7.3,
 'revenue_usd_m': 27.82,
 'cost_usd_m': 20.52,
 'advertisers_served': 4171,
 'advertisers_abandoned': 0,
 'worst_month_avg_wait_days': 0.0,
 'still_waiting_end_of_year': 0}

In [10]:
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import SystemMessage

SYSTEM = SystemMessage(content=(
    "You are a staffing planning assistant for Google Ads advertiser support in 22 countries for 2026. "
    "Answer ONLY using numbers returned by your tools; never calculate or estimate numbers yourself. "
    "If a question needs numbers you don't have, call a tool. "
    "Always say which scenario (base, low, high) your numbers are for. "
    "Write money as $X.XM. Keep answers short and clear for a manager. "
    "If asked something outside staffing for these 22 countries, say you can't help with that."
))

llm_with_tools = llm.bind_tools(tools)

def assistant(state: MessagesState):
    return {"messages": [llm_with_tools.invoke([SYSTEM] + state["messages"])]}

graph = StateGraph(MessagesState)
graph.add_node("assistant", assistant)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "assistant")
graph.add_conditional_edges("assistant", tools_condition)
graph.add_edge("tools", "assistant")
agent = graph.compile()
print("Agent ready")

Agent ready


In [12]:
def ask(question):
    result = agent.invoke({"messages": [("user", question)]})
    for msg in result["messages"]:
        for call in getattr(msg, "tool_calls", []) or []:
            print(f"🔧 {call['name']}({call['args']})")
    print("\n" + result["messages"][-1].content)

ask("What happens if France keeps 150 agents all year instead of the recommended plan?")

🔧 evaluate_plan({'country': 'France', 'agents_jan_sep': 150, 'agents_oct_dec': 150})

If France keeps 150 agents all year, the results would be:

- Net Value: $7.3M
- Revenue: $27.82M
- Cost: $20.52M
- Advertisers Served: 4,171
- Advertisers Abandoned: 0
- Worst Month Average Wait Days: 0.0
- Still Waiting End of Year: 0

This is based on the base scenario.


In [13]:
ask("Is Vietnam's plan still OK if sign-ups are 15% higher? If not, what should it be?")
ask("Which three countries gain the most from the recommended plan?")
ask("Why does UAE get zero agents?")
ask("What is the weather in Paris?")

🔧 reoptimize({'country': 'Vietnam', 'scenario': 'high'})
🔧 get_recommended_plan({'country': 'Vietnam', 'scenario': 'base'})

Vietnam's current plan is not sufficient for a 15% increase in sign-ups. The recommended headcount for the high scenario is:

- Agents (Jan-Sep): 88
- Agents (Oct-Dec): 123

This plan would yield a net value of $11.23M.
🔧 all_countries_overview({})

The three countries that gain the most from the recommended plan are:

1. **USA**: $15.7M
2. **Australia**: $8.0M
3. **UK**: $12.4M

This is based on the base scenario.

I can't help with that.

I can't help with that.


In [ ]:
sh